# T8 It2: audit of the earlier methods: fitted to the data? real gain or just more answers?

**Team review (2026-11-14):** the T7 judge was trained on the data it was meant to improve, so it fits that data; the final data will not be the
same. Every earlier method is checked the same way here.

**Protocol (locked):** official is cut by place into a TRY half (185 photos) and a TEST half (194), heights balanced (It0 split,
`results/official_split.csv`, block `train/checks/holdout.py`). This notebook never loads the TEST half. Design data = the five NUDT sets.

**Two questions per method:**
1. *Fitted?* The same method built with official in its training/design vs built on NUDT only, both scored on official TRY.
2. *Real gain or more answers?* A method that answers more photos is compared with the **plain inlier bar that answers the same number of photos**
   (`versus_frontier`): sitting on that curve means it only answers more; above it means it judges better. `answer_change` splits the gain into
   answers added (right / wrong), answers removed and wrong answers.

In [1]:
import os, sys, time, json
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd, joblib
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
from train.checks.candidate_judge import (FEATURES_CTX, TAUS, baseline, decide, judge_new, judge_probability, ltt_tau, scan_tau, summary)
from train.checks.holdout import NUDT_DESIGN, answer_change, bar_frontier, load_t8_table, versus_frontier
from train.checks.two_tier import null_decide, null_pvalues
RES = ROOT / "tests/Test_Iteration_8/results"
table = load_t8_table(ROOT, locked=True)          # official_test is not loaded here
SEEN = ["official_try"] + NUDT_DESIGN
assert "official_test" not in set(table.data)
table.groupby("data").photo.nunique()

data
nudt101         300
nudt202         200
nudt404         200
nudt505         300
nudt606         300
official_try    185
Name: photo, dtype: int64

## 1. The plain-bar curve (the reference)
PC: bar b on the 6 standard queries, b + 3 on all 27. Jetson: bar b, second pass, cap 10, M = 5.

In [2]:
t0 = time.time(); FRONT = bar_frontier(table); FRONT.to_csv(RES / "it2_bar_frontier.csv", index=False); print("%.0f s" % (time.time() - t0))
for var in ["pc", "jetson"]:
    f = FRONT[FRONT["var"] == var]
    display(pd.concat({"score": f.pivot(index="bar", columns="data", values="score"), "wrong": f.pivot(index="bar", columns="data", values="wrong")}, axis=1).round(2))

23 s


score                                                wrong                                             
data nudt101 nudt202 nudt404 nudt505 nudt606 official_try nudt101 nudt202 nudt404 nudt505 nudt606 official_try
bar                                                                                                           
3      47.24   55.62   48.51   50.02   51.20        55.20      19       3      12      15      13            1
4      49.04   56.52   49.11   54.22   52.72        54.81      15       2      11       6      10            0
5      51.84   57.12   51.81   56.02   54.12        53.84       9       1       7       2       7            0
6      54.44   55.86   52.32   55.48   55.12        52.86       2       2       6       2       4            0
7      54.32   55.56   53.52   54.96   55.92        51.70       2       2       4       1       2            0
8      54.84   57.06   54.24   54.24   55.08        51.37       0       0       2       1       2            0
9      54.52   56.76   54.54   54.64   55.88        50.40       0       0       1       0       0            0
10     54.32   56.46   54.54   54.44   55.88        50.40       0       0       0       0       0            0
11     53.92   56.28   54.66   54.24   55.88        49.75       0       0       0       0       0            0
12     53.92   55.68   54.66   53.84   55.68        49.43       0       0       0       0       0            0
13     53.52   54.78   54.36   53.32   55.48        49.43       0       0       0       0       0            0
14     53.52   54.18   54.36   52.52   54.14        49.10       0       0       0       0       0            0
15     52.72   53.88   53.76   51.32   53.54        47.16       0       0       0       0       0            0

score                                                wrong                                             
data nudt101 nudt202 nudt404 nudt505 nudt606 official_try nudt101 nudt202 nudt404 nudt505 nudt606 official_try
bar                                                                                                           
3      49.36   51.54   50.31   51.64   50.74        54.88      15       8       9      11      13            1
4      50.36   53.52   51.81   54.04   51.54        55.00      12       5       7       5      11            0
5      52.76   55.92   54.09   55.44   53.14        52.86       6       1       4       2       7            0
6      55.00   54.66   52.92   55.36   53.94        52.54       1       2       5       1       4            0
7      54.56   54.66   54.42   54.88   54.74        51.70       1       1       3       1       2            0
8      54.24   55.26   55.44   54.68   54.42        51.37       0       0       0       1       2            0
9      54.12   54.96   55.44   54.36   54.02        50.08       0       0       0       0       2            0
10     53.92   54.66   54.84   53.76   54.62        49.43       0       0       0       0       0            0
11     53.72   54.66   54.96   53.44   54.42        49.10       0       0       0       0       0            0
12     53.12   54.36   54.96   53.24   54.22        48.78       0       0       0       0       0            0
13     52.72   54.06   54.66   52.72   53.82        48.78       0       0       0       0       0            0
14     52.52   53.58   54.36   52.32   53.48        48.78       0       0       0       0       0            0
15     52.12   52.98   53.76   51.72   53.28        47.48       0       0       0       0       0            0

**Read:** on official TRY a bar of 4 makes **0 wrong answers** and scores 54.8 (PC), +3.4 over bar 8; on NUDT the same bar makes 6-15 wrong
answers per set. So on official, *answering more is almost free*, and any method that answers more will look good there. The decision problem is
set by how many look-alike places a map has, which differs between data sets.

## 2. The T7 judge: trained with official vs on NUDT only

In [3]:
frozen = joblib.load(ROOT / "tests/Test_Iteration_7/results/judge_t7.joblib")
frozen = frozen["model"] if isinstance(frozen, dict) else frozen
p_frozen = judge_new(frozen, table)                                   # trained on official (all 379) + NUDT 101-505: in-sample on TRY
p_nudt, _ = judge_probability(table, NUDT_DESIGN, FEATURES_CTX, "gbm")   # trained on the five NUDT sets only (out-of-fold on NUDT)
KW = dict(override=12, min_choice=3, wide_tau=0.95)
tau_pc = max(ltt_tau(scan_tau(table, p_nudt, TAUS, **KW), NUDT_DESIGN, 0.005), 0.9)
tau_jet = max(ltt_tau(scan_tau(table, p_nudt, TAUS, jetson=True, m=5, **KW), NUDT_DESIGN, 0.005), 0.9)
print("cuts chosen on NUDT only: PC %.3f, Jetson %.3f" % (tau_pc, tau_jet))
M = {"pc": {}, "jetson": {}}
M["pc"]["bar 8 / 11 (base)"] = baseline(table, 8, 11, min_choice=3)
M["jetson"]["Jetson today (base)"] = baseline(table, 8, 8, jetson=True, m=5)
M["pc"]["T7 judge, trained with official (frozen)"] = decide(table, p_frozen, 0.9, **KW)
M["pc"]["T7 judge, NUDT only"] = decide(table, p_nudt, tau_pc, **KW)
M["jetson"]["T7 judge, NUDT only"] = decide(table, p_nudt, tau_jet, jetson=True, m=5, **KW)
M["jetson"]["T7 judge, NUDT only + early exit 0.98"] = decide(table, p_nudt, tau_jet, jetson=True, m=5, early_tau=0.98, **KW)
M["pc"]["T7 judge, NUDT only, min 4 inliers"] = decide(table, p_nudt, tau_pc, **{**KW, "min_choice": 4})
veto = table.copy(); veto.loc[(veto.z < 1.3) & (veto.inliers < 15), "inliers"] = 0      # T6 DINO-z veto: such candidates can be neither chosen nor answered
M["pc"]["T6 z-veto on bar 8 / 11"] = baseline(veto, 8, 11, min_choice=3)
M["jetson"]["T6 z-veto on Jetson today"] = baseline(veto, 8, 8, jetson=True, m=5)
pp = null_pvalues(table, "inliers")
M["pc"]["null-calibrated inlier cut, eps 0.05 / 0.017"] = null_decide(pp, 0.05, 0.05 / 3)
table["pj"] = p_nudt
ppj = null_pvalues(table, "pj")
M["pc"]["null-calibrated judge cut, eps 0.02 / 0.007"] = null_decide(ppj, 0.02, 0.02 / 3)
rows = []
for var, ms in M.items():
    base = list(ms.values())[0]
    for name, r in ms.items():
        v = versus_frontier(r, FRONT, var).set_index("data")
        a = answer_change(base, r).set_index("data")
        for d in SEEN:
            rows.append(dict(var=var, method=name, data=d, **v.loc[d].to_dict(), **a.loc[d, ["added_right", "added_wrong", "removed", "removed_wrong"]].to_dict()))
AUDIT = pd.DataFrame(rows); AUDIT.to_csv(RES / "it2_audit.csv", index=False)
for d in SEEN:
    print("==", d); display(AUDIT[AUDIT.data == d].drop(columns="data").set_index(["var", "method"]).round(2))

cuts chosen on NUDT only: PC 0.900, Jetson 0.925


== official_try


score  wrong  n_answered  queries  same_n_bar  bar_score  bar_wrong  above_bar  added_right  added_wrong  removed  removed_wrong
var    method                                                                                                                                                                        
pc     bar 8 / 11 (base)                             51.37    0.0       162.0     8.95         8.0      51.37        0.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, trained with official (frozen)      56.95    0.0       180.0     7.36         3.0      55.20        1.0       1.75         18.0          0.0      0.0            0.0
       T7 judge, NUDT only                           54.03    0.0       171.0     8.27         5.0      53.84        0.0       0.19          9.0          0.0      0.0            0.0
       T7 judge, NUDT only, min 4 inliers            54.03    0.0       171.0     8.27         5.0      53.84        0.0       0.19          9.0          0.0      0.0            0.0
       T6 z-veto on bar 8 / 11                       51.37    0.0       162.0     8.95         8.0      51.37        0.0       0.00          0.0          0.0      0.0            0.0
       null-calibrated inlier cut, eps 0.05 / 0.017  51.37    0.0       162.0     0.00         8.0      51.37        0.0       0.00          0.0          0.0      0.0            0.0
       null-calibrated judge cut, eps 0.02 / 0.007   55.85    0.0       177.0     0.00         3.0      55.20        1.0       0.65         15.0          0.0      0.0            0.0
jetson Jetson today (base)                           51.37    0.0       162.0     6.53         8.0      51.37        0.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, NUDT only                           52.54    0.0       166.0     6.42         6.0      52.54        0.0       0.00          4.0          0.0      0.0            0.0
       T7 judge, NUDT only + early exit 0.98         51.31    0.0       166.0     2.43         6.0      52.54        0.0      -1.23          4.0          0.0      0.0            0.0
       T6 z-veto on Jetson today                     51.37    0.0       162.0     6.53         8.0      51.37        0.0       0.00          0.0          0.0      0.0            0.0

== nudt101


score  wrong  n_answered  queries  same_n_bar  bar_score  bar_wrong  above_bar  added_right  added_wrong  removed  removed_wrong
var    method                                                                                                                                                                        
pc     bar 8 / 11 (base)                             54.84    0.0       275.0     8.45         8.0      54.84        0.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, trained with official (frozen)      55.52    0.0       278.0     8.24         7.0      54.32        2.0       1.20          4.0          0.0      1.0            0.0
       T7 judge, NUDT only                           55.52    0.0       278.0     8.17         7.0      54.32        2.0       1.20          4.0          0.0      1.0            0.0
       T7 judge, NUDT only, min 4 inliers            55.52    0.0       278.0     8.17         7.0      54.32        2.0       1.20          4.0          0.0      1.0            0.0
       T6 z-veto on bar 8 / 11                       54.84    0.0       275.0     8.45         8.0      54.84        0.0       0.00          0.0          0.0      0.0            0.0
       null-calibrated inlier cut, eps 0.05 / 0.017  52.80    0.0       264.0     0.00        15.0      52.72        0.0       0.08          0.0          0.0     11.0            0.0
       null-calibrated judge cut, eps 0.02 / 0.007   54.36    2.0       279.0     0.00         6.0      54.44        2.0      -0.08          4.0          1.0      1.0            0.0
jetson Jetson today (base)                           54.24    0.0       272.0     6.43         8.0      54.24        0.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, NUDT only                           54.52    0.0       273.0     6.40         8.0      54.24        0.0       0.28          4.0          0.0      3.0            0.0
       T7 judge, NUDT only + early exit 0.98         54.44    0.0       273.0     2.15         8.0      54.24        0.0       0.20          4.0          0.0      3.0            0.0
       T6 z-veto on Jetson today                     54.24    0.0       272.0     6.43         8.0      54.24        0.0       0.00          0.0          0.0      0.0            0.0

== nudt202


... (3 outputs left out)

== nudt505


score  wrong  n_answered  queries  same_n_bar  bar_score  bar_wrong  above_bar  added_right  added_wrong  removed  removed_wrong
var    method                                                                                                                                                                        
pc     bar 8 / 11 (base)                             54.24    1.0       275.0     8.38         8.0      54.24        1.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, trained with official (frozen)      56.84    0.0       285.0     7.82         6.0      55.48        2.0       1.36         11.0          0.0      1.0            1.0
       T7 judge, NUDT only                           56.76    0.0       285.0     7.82         6.0      55.48        2.0       1.28         11.0          0.0      1.0            1.0
       T7 judge, NUDT only, min 4 inliers            56.76    0.0       285.0     7.89         6.0      55.48        2.0       1.28         11.0          0.0      1.0            1.0
       T6 z-veto on bar 8 / 11                       54.24    1.0       275.0     8.38         8.0      54.24        1.0       0.00          0.0          0.0      0.0            0.0
       null-calibrated inlier cut, eps 0.05 / 0.017  54.24    0.0       272.0     0.00        11.0      54.24        0.0       0.00          0.0          0.0      3.0            1.0
       null-calibrated judge cut, eps 0.02 / 0.007   55.96    2.0       287.0     0.00         5.0      56.02        2.0      -0.06         12.0          1.0      1.0            1.0
jetson Jetson today (base)                           54.68    1.0       278.0     6.39         8.0      54.68        1.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, NUDT only                           55.56    0.0       279.0     6.33         7.0      54.88        1.0       0.68          3.0          0.0      2.0            1.0
       T7 judge, NUDT only + early exit 0.98         55.32    0.0       279.0     1.99         7.0      54.88        1.0       0.44          3.0          0.0      2.0            1.0
       T6 z-veto on Jetson today                     55.08    0.0       277.0     6.39         8.0      54.68        1.0       0.40          0.0          0.0      1.0            1.0

== nudt606


score  wrong  n_answered  queries  same_n_bar  bar_score  bar_wrong  above_bar  added_right  added_wrong  removed  removed_wrong
var    method                                                                                                                                                                        
pc     bar 8 / 11 (base)                             55.08    2.0       285.0     7.54         8.0      55.08        2.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, trained with official (frozen)      55.80    2.0       289.0     7.54         7.0      55.92        2.0      -0.12          4.0          1.0      1.0            1.0
       T7 judge, NUDT only                           56.40    1.0       289.0     7.54         7.0      55.92        2.0       0.48          5.0          0.0      1.0            1.0
       T7 judge, NUDT only, min 4 inliers            56.40    1.0       289.0     7.54         7.0      55.92        2.0       0.48          5.0          0.0      1.0            1.0
       T6 z-veto on bar 8 / 11                       55.48    1.0       284.0     7.61         8.0      55.08        2.0       0.40          0.0          0.0      1.0            1.0
       null-calibrated inlier cut, eps 0.05 / 0.017  55.88    0.0       283.0     0.00         9.0      55.88        0.0       0.00          0.0          0.0      2.0            2.0
       null-calibrated judge cut, eps 0.02 / 0.007   56.80    1.0       290.0     0.00         7.0      55.92        2.0       0.88          7.0          0.0      2.0            1.0
jetson Jetson today (base)                           54.42    2.0       280.0     6.29         8.0      54.42        2.0       0.00          0.0          0.0      0.0            0.0
       T7 judge, NUDT only                           55.02    1.0       280.0     6.30         8.0      54.42        2.0       0.60          1.0          0.0      1.0            1.0
       T7 judge, NUDT only + early exit 0.98         54.70    1.0       280.0     1.89         8.0      54.42        2.0       0.28          1.0          0.0      1.0            1.0
       T6 z-veto on Jetson today                     54.42    2.0       280.0     6.31         8.0      54.42        2.0       0.00          0.0          0.0      0.0            0.0

**Read (official TRY):**
- *Fitted:* the T7 judge that saw official scores ~57 on TRY; the same architecture trained on NUDT only scores ~54. About **3 points of the T7
  official number came from fitting official itself** (in-sample), which supports the review.
- *More answers:* trained on NUDT only, the judge sits **on the plain-bar curve on official** (about +0.2 over the bar answering as many photos):
  there it only answers more. On NUDT it sits above the curve (+1 to +3) because it removes the bars' 1-2 wrong answers per set, but it adds a
  new wrong answer on NUDT 202 and keeps one on 606.
- The Jetson early exit costs ~1.2 points on official TRY (answers taken before the most precise query) for ~60 % less matching time.
- The T6 DINO-z veto only removes wrong answers (404, 505, 606) and costs nothing on TRY: a safety net, not a gain.
- The label-free null-calibrated cut (idea: set the cut from each set's own far candidates, so look-alike-rich maps get a stricter cut without
  labels) is safe with the inlier score but scores below today's bar, and with the judge's score it lets 1-2 wrong answers into NUDT: dropped.

## 3. Precision: other position estimates of the chosen place (answers at >= 8 inliers within 8 sigma)

In [4]:
import glob
truth = {}
for d, folder in [("official_try", "official")] + [(s, {"nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404", "nudt505": "nudt_fresh505", "nudt606": "nudt_fresh606"}[s]) for s in NUDT_DESIGN]:
    for f in glob.glob(str(ROOT / f"tests/Test_Iteration_6/results/rich_dump/{folder}/photo_*.npz")):
        truth[(d, int(f.split("_")[-1][:-4]))] = tuple(np.load(f, allow_pickle=True)["truth"][:2])
rows = []
for (d, p), g in table.groupby(["data", "photo"], sort=False):
    g = g[g.inliers >= 3]
    if not len(g) or (d, p) not in truth: continue
    c = g.iloc[int(np.lexsort((-g.coarse.to_numpy(), -(g.inliers + g.z).to_numpy()))[0])]
    if c.inliers < 8 or c.err_sigma > 8: continue
    tx, ty = truth[(d, p)]; near = g[np.hypot(g.x - c.x, g.y - c.y) <= 2 * c.sigma]; w = near.inliers.to_numpy()
    est = {"similarity fit (today)": (c.x, c.y), "translation fit": (c.tx, c.ty) if c.inl_t3 > 0 else (c.x, c.y),
           "inlier-weighted mean of the place over queries": ((near.x * w).sum() / w.sum(), (near.y * w).sum() / w.sum()),
           "largest-hull candidate of the place": tuple(near.loc[near.hull.idxmax(), ["x", "y"]])}
    rows.append(dict(data=d, **{k: np.hypot(x - tx, y - ty) / c.sigma for k, (x, y) in est.items()}))
E = pd.DataFrame(rows)
pts = lambda v: np.select([v <= 1, v <= 2, v <= 4, v <= 8], [10, 6, 3, 0], -20).mean()
E.groupby("data").agg(pts).round(3)

,similarity fit (today),translation fit,inlier-weighted mean of the place over queries,largest-hull candidate of the place
data,,,,
nudt101,9.971,9.884,9.957,9.986
nudt202,9.907,9.824,9.865,9.865
nudt404,9.957,9.872,10.000,9.957
nudt505,9.904,9.791,9.918,9.904
nudt606,9.885,9.791,9.875,9.889
official_try,9.806,9.176,9.806,9.733


**Read:** today's similarity-fit position is already the best estimate on every set (official TRY 9.81 of 10 per answer); no change here.